In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Problem statement
Amazon is expanding its AI-powered customer service ecosystem by combining cloud infrastructure, generative AI capabilities, automation, and enterprise AI services. This strategy creates opportunities for efficiency, customer experience improvement, and new market growth, but it also introduces uncertainty across technology reliability, cybersecurity, operations, compliance, finance, and governance. This risk assessment uses the Group 3 Amazon Risk Register to score identified risks by likelihood and impact, classify risk severity, and visualize the results in a risk heat map for ERM decision-making.

In [ ]:
# Load the Amazon Risk Register workbook
# Keep the Excel file in the same folder as this notebook when running locally.

file_path = "ALY6130_Risk Register_Group3_Amazon.xlsx"

# Read the Risk Register sheet from the workbook
df_risk_register = pd.read_excel(file_path, sheet_name="RiskRegister")

# Preview the first few rows
df_risk_register.head()

In [ ]:
# Select and rename relevant columns
risk_df = df_risk_register[['Risk #', 'The Risk of/That', 'Likelihood Score', 'Impact Score']].dropna()
risk_df.columns = ['Risk_ID', 'Risk_Description', 'Likelihood_Score', 'Impact_Score']

# Make sure scores are numeric
risk_df['Risk_ID'] = risk_df['Risk_ID'].astype(int)
risk_df['Likelihood_Score'] = pd.to_numeric(risk_df['Likelihood_Score'], errors='coerce')
risk_df['Impact_Score'] = pd.to_numeric(risk_df['Impact_Score'], errors='coerce')

# Calculate total risk score
risk_df['Risk_Score'] = risk_df['Likelihood_Score'] * risk_df['Impact_Score']

# Define severity label based on total risk score
# These thresholds align with the 1, 3, 5, 7, 9 scoring scale used in the Risk Calculation Sheet.
def classify_severity(score):
    if score >= 56:
        return 'High'
    elif score >= 28:
        return 'Medium'
    else:
        return 'Low'

# Apply function to classify severity
risk_df['Severity_Label'] = risk_df['Risk_Score'].apply(classify_severity)

# Create label used in charts
risk_df['Label'] = 'R' + risk_df['Risk_ID'].astype(str)

# View cleaned data
risk_df.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# This model uses a Random Forest classifier to predict risk severity levels based on likelihood and impact scores.
# The model is used as a supporting analytical exercise; the main ERM score still comes from the Risk Register.
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

In [ ]:
# Re-select features and target: defines the feature matrix (X) using Likelihood and Impact scores,
# and creates the target variable (y) by assigning severity labels based on rule-based thresholds.

X = risk_df[['Likelihood_Score', 'Impact_Score']]
y = risk_df['Severity_Label']

# Check class distribution before model evaluation
print("Severity class distribution:")
print(y.value_counts())

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Use stratified cross-validation because the dataset is small but contains multiple severity classes.
# This gives a more stable evaluation than a single train-test split.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print(f"Mean accuracy (5-fold CV): {scores.mean():.2f}")
print("Accuracy for each fold:", scores)

### Note:
The model results should be interpreted with caution because the dataset contains only 37 risks. The Random Forest model is used mainly to show how likelihood and impact scores can be translated into severity categories. The risk assessment itself should still be justified using external evidence, industry research, and the documented business context.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

# Cross-validated predictions
y_pred_cv = cross_val_predict(model, X, y, cv=cv)

# Classification metrics
print("Classification Report:")
print(classification_report(y, y_pred_cv))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y, y_pred_cv, labels=['Low', 'Medium', 'High']))

The model is expected to perform well because severity is directly derived from likelihood and impact scores. However, this does not replace risk reasoning. The most important part of the assignment is to support each score with credible evidence such as industry reports, market trends, competitor activity, cybersecurity incidents, AI adoption challenges, and operational examples related to Amazon’s AI ecosystem expansion.

In [ ]:
# Create the risk heat map using the exact likelihood and impact values from the Risk Register.
# X-axis = Likelihood Score
# Y-axis = Impact Score

severity_map = risk_df.copy()

# Convert severity labels to numeric values for heatmap coloring
severity_levels = {'Low': 1, 'Medium': 2, 'High': 3}
severity_map['Severity_Num'] = severity_map['Severity_Label'].map(severity_levels)

# Pivot table with most severe risk per likelihood-impact cell
pivot_severity = severity_map.groupby(['Impact_Score', 'Likelihood_Score'])['Severity_Num'].max().unstack()
pivot_severity = pivot_severity.sort_index(ascending=False)
pivot_severity = pivot_severity[pivot_severity.columns.sort_values()]

# Annotation table showing Risk IDs inside each cell
pivot_labels = severity_map.groupby(['Impact_Score', 'Likelihood_Score'])['Label'].apply(lambda x: ', '.join(x)).unstack()
pivot_labels = pivot_labels.reindex(index=pivot_severity.index, columns=pivot_severity.columns)
pivot_labels = pivot_labels.fillna('')

plt.figure(figsize=(10, 6))
sns.heatmap(
    pivot_severity,
    annot=pivot_labels,
    fmt='',
    cmap=sns.color_palette(['#8BC34A', '#FFD966', '#E06666'], as_cmap=True),
    linewidths=0.8,
    linecolor='white',
    cbar_kws={'label': 'Severity Level: 1 = Low, 2 = Medium, 3 = High'},
    vmin=1,
    vmax=3
)

plt.title('Amazon AI-Powered Customer Service Expansion Risk Heat Map', fontsize=14, fontweight='bold')
plt.xlabel('Likelihood Score')
plt.ylabel('Impact Score')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

# Reuse model and labels
y_pred_cv = cross_val_predict(model, X, y, cv=cv)
conf_matrix = confusion_matrix(y, y_pred_cv, labels=['Low', 'Medium', 'High'])

# Plot confusion matrix as heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Low', 'Medium', 'High'],
    yticklabels=['Low', 'Medium', 'High']
)
plt.title("Confusion Matrix (Cross-Validated)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

### Interpretation
The heat map shows how Amazon-related risks are distributed across the likelihood and impact scoring scale. High-risk cells contain risks with both strong probability and significant business impact. These should receive the highest priority in ERM planning, especially risks connected to cybersecurity, data privacy, cloud service reliability, AI integration, and delivery disruption.

The confusion matrix shows how consistently the model reproduces the severity labels derived from the risk score thresholds. Since the target labels are rule-based, the model is mainly a programmatic validation and visualization tool rather than a replacement for management judgment.

In [ ]:
# Summarize risk counts and top risks by total score
summary_table = risk_df.groupby('Severity_Label').agg(
    Risk_Count=('Risk_ID', 'count'),
    Average_Likelihood=('Likelihood_Score', 'mean'),
    Average_Impact=('Impact_Score', 'mean'),
    Average_Risk_Score=('Risk_Score', 'mean')
).reset_index()

summary_table

In [ ]:
# Display the top 10 highest-scoring risks for final discussion and prioritization

top_10_risks = risk_df.sort_values(by='Risk_Score', ascending=False).head(10)
top_10_risks[['Risk_ID', 'Risk_Description', 'Likelihood_Score', 'Impact_Score', 'Risk_Score', 'Severity_Label']]